In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

data_dir = 'preprocessed'
subsets = ['train', 'valid', 'test']

lower_red1, upper_red1 = np.array([0, 50, 50]), np.array([10, 255, 255])
lower_red2, upper_red2 = np.array([170, 50, 50]), np.array([180, 255, 255])
lower_blue, upper_blue = np.array([100, 50, 50]), np.array([130, 255, 255])
lower_green, upper_green = np.array([40, 50, 50]), np.array([80, 255, 255])

existence_threshold = 50

# Create directories for extracted parts
shell_output_dir = os.path.join(data_dir, 'extracted', 'shell')
head_output_dir = os.path.join(data_dir, 'extracted', 'head')
limbs_output_dir = os.path.join(data_dir, 'extracted', 'limbs')
os.makedirs(shell_output_dir, exist_ok=True)
os.makedirs(head_output_dir, exist_ok=True)
os.makedirs(limbs_output_dir, exist_ok=True)

for subset in subsets:
    mask_dir = os.path.join(data_dir, subset, 'masks')
    image_dir = os.path.join(data_dir, subset, 'images')
    mask_files = sorted(os.listdir(mask_dir))
    image_files = sorted(os.listdir(image_dir))

    # Check if both directories have the same number of files

    if len(mask_files) != len(image_files):
        print(f"Warning: Number of masks and images do not match in {subset} subset.")
        continue

    valid_count = 0
    invalid_count = 0

    for mask_file, image_file in zip(mask_files, image_files):
        mask_path = os.path.join(mask_dir, mask_file)
        image_path = os.path.join(image_dir, image_file)
        
        mask = cv2.imread(mask_path)
        image = cv2.imread(image_path)
        
        if mask is None or image is None:
            print(f"Error reading {mask_file} or {image_file}, skipping.")
            continue

        mask_hsv = cv2.cvtColor(mask, cv2.COLOR_BGR2HSV)
        
        red_mask1 = cv2.inRange(mask_hsv, lower_red1, upper_red1)
        red_mask2 = cv2.inRange(mask_hsv, lower_red2, upper_red2)
        red_mask = cv2.bitwise_or(red_mask1, red_mask2)  
        
        blue_mask = cv2.inRange(mask_hsv, lower_blue, upper_blue)
        green_mask = cv2.inRange(mask_hsv, lower_green, upper_green)
        
        red_count = cv2.countNonZero(red_mask)
        blue_count = cv2.countNonZero(blue_mask)
        green_count = cv2.countNonZero(green_mask)
        
        if red_count > existence_threshold and blue_count > existence_threshold and green_count > existence_threshold:
            shell = cv2.bitwise_and(image, image, mask=red_mask)
            head = cv2.bitwise_and(image, image, mask=blue_mask)
            limbs = cv2.bitwise_and(image, image, mask=green_mask)

            # Save extracted parts to their respective directories
            cv2.imwrite(os.path.join(shell_output_dir, f'shell_{mask_file}'), shell)
            cv2.imwrite(os.path.join(head_output_dir, f'head_{mask_file}'), head)
            cv2.imwrite(os.path.join(limbs_output_dir, f'limbs_{mask_file}'), limbs)
            valid_count += 1
        else:
            invalid_count += 1

    print(f"Subset: {subset}, Valid Images: {valid_count}, Invalid Images: {invalid_count}")



In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

data_dir = 'preprocessed'
subsets = ['train', 'valid', 'test']

lower_red1, upper_red1 = np.array([0, 50, 50]), np.array([10, 255, 255])
lower_red2, upper_red2 = np.array([170, 50, 50]), np.array([180, 255, 255])
lower_blue, upper_blue = np.array([100, 50, 50]), np.array([130, 255, 255])
lower_green, upper_green = np.array([40, 50, 50]), np.array([80, 255, 255])

existence_threshold = 500 
for subset in subsets:
    mask_dir = os.path.join(data_dir, subset, 'masks')
    image_dir = os.path.join(data_dir, subset, 'images')
    mask_files = sorted(os.listdir(mask_dir))
    image_files = sorted(os.listdir(image_dir))
    
    for mask_file, image_file in zip(mask_files, image_files):
        mask_path = os.path.join(mask_dir, mask_file)
        image_path = os.path.join(image_dir, image_file)
        
        mask = cv2.imread(mask_path)
        image = cv2.imread(image_path)
        mask_hsv = cv2.cvtColor(mask, cv2.COLOR_BGR2HSV)
        red_mask1 = cv2.inRange(mask_hsv, lower_red1, upper_red1)
        red_mask2 = cv2.inRange(mask_hsv, lower_red2, upper_red2)
        red_mask = cv2.bitwise_or(red_mask1, red_mask2)  
        blue_mask = cv2.inRange(mask_hsv, lower_blue, upper_blue)
        green_mask = cv2.inRange (mask_hsv, lower_green, upper_green)
        red_count = cv2.countNonZero(red_mask)
        blue_count = cv2.countNonZero(blue_mask)
        green_count = cv2.countNonZero(green_mask)
        if red_count > existence_threshold and blue_count > existence_threshold and green_count > existence_threshold:
            shell = cv2.bitwise_and(image, image, mask=red_mask)
            head = cv2.bitwise_and(image, image, mask=blue_mask)
            limbs = cv2.bitwise_and(image, image, mask=green_mask)
            plt.figure(figsize=(12, 4))
            plt.subplot(1, 4, 1)
            plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            plt.title('Original Image')
            
            plt.subplot(1, 4, 2)
            plt.imshow(cv2.cvtColor(shell, cv2.COLOR_BGR2RGB))
            plt.title('Shell (Red)')
            
            plt.subplot(1, 4, 3)
            plt.imshow(cv2.cvtColor(head, cv2.COLOR_BGR2RGB))
            plt.title('Head (Blue)')
            
            plt.subplot(1, 4, 4)
            plt.imshow(cv2.cvtColor(limbs, cv2.COLOR_BGR2RGB))
            plt.title('Limbs (Green)')
            
            plt.show()

            output_dir = os.path.join(data_dir, subset, 'extracted')
            os.makedirs(output_dir, exist_ok=True)
            
            cv2.imwrite(os.path.join(output_dir, f'shell_{mask_file}'), shell)
            cv2.imwrite(os.path.join(output_dir, f'head_{mask_file}'), head)
            cv2.imwrite(os.path.join(output_dir, f'limbs_{mask_file}'), limbs)
        else:
            print(f"{mask_file} does not contain all required parts, skipping.")

In [ ]:
from skimage.feature import hog

# Directories for saving features
data_dir = 'preprocessed'
shell_output_dir = os.path.join(data_dir, 'extracted', 'shell')
head_output_dir = os.path.join(data_dir, 'extracted', 'head')
limbs_output_dir = os.path.join(data_dir, 'extracted', 'limbs')

shell_features_dir = os.path.join(data_dir, 'features', 'shell')
head_features_dir = os.path.join(data_dir, 'features', 'head')
limbs_features_dir = os.path.join(data_dir, 'features', 'limbs')
os.makedirs(shell_features_dir, exist_ok=True)
os.makedirs(head_features_dir, exist_ok=True)
os.makedirs(limbs_features_dir, exist_ok=True)

# Function to extract HOG features and shape and save them
def extract_hog_features(image_path, feature_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        features, _ = hog(img, orientations=12, pixels_per_cell=(5, 5),
                          cells_per_block=(2, 2), block_norm='L2-Hys', visualize=True)
        np.save(feature_path, features)
    print("Extracted HOG features:", features)

# Extract features for shell, head, and limbs
for img_file in os.listdir(shell_output_dir):
    shell_path = os.path.join(shell_output_dir, img_file)
    feature_path = os.path.join(shell_features_dir, f'shell_{img_file}.npy')
    extract_hog_features(shell_path, feature_path)

for img_file in os.listdir(head_output_dir):
    head_path = os.path.join(head_output_dir, img_file)
    feature_path = os.path.join(head_features_dir, f'head_{img_file}.npy')
    extract_hog_features(head_path, feature_path)

for img_file in os.listdir(limbs_output_dir):
    limbs_path = os.path.join(limbs_output_dir, img_file)
    feature_path = os.path.join(limbs_features_dir, f'limbs_{img_file}.npy')
    extract_hog_features(limbs_path, feature_path)

print("HOG feature extraction completed for all parts.")


In [ ]:
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
# Load features for each part
shell_features = [np.load(os.path.join(shell_features_dir, f)) for f in os.listdir(shell_features_dir)]
head_features = [np.load(os.path.join(head_features_dir, f)) for f in os.listdir(head_features_dir)]
limbs_features = [np.load(os.path.join(limbs_features_dir, f)) for f in os.listdir(limbs_features_dir)]

# Create labels for each part
shell_labels = ['Shell'] * len(shell_features)
head_labels = ['Head'] * len(head_features)
limbs_labels = ['Limbs'] * len(limbs_features)

# Combine features and labels for each part
X_shell, y_shell = np.array(shell_features), np.array(shell_labels)
X_head, y_head = np.array(head_features), np.array(head_labels)
X_limbs, y_limbs = np.array(limbs_features), np.array(limbs_labels)

# Split the data into training and testing sets
X_train_shell, X_test_shell, y_train_shell, y_test_shell = train_test_split(X_shell, y_shell, test_size=0.2, random_state=42)
X_train_head, X_test_head, y_train_head, y_test_head = train_test_split(X_head, y_head, test_size=0.2, random_state=42)
X_train_limbs, X_test_limbs, y_train_limbs, y_test_limbs = train_test_split(X_limbs, y_limbs, test_size=0.2, random_state=42)

print("Data splitting completed for all parts.")

In [ ]:
# Train KNN classifiers for each part
knn_shell_classifier = KNeighborsClassifier(n_neighbors=3)
knn_head_classifier = KNeighborsClassifier(n_neighbors=3)
knn_limbs_classifier = KNeighborsClassifier(n_neighbors=3)

# Train models if there are multiple classes in the training set
knn_shell_classifier.fit(X_train_shell, y_train_shell)
knn_head_classifier.fit(X_train_head, y_train_head)
knn_limbs_classifier.fit(X_train_limbs, y_train_limbs)

# Evaluate the classifiers
knn_shell_pred = knn_shell_classifier.predict(X_test_shell)
knn_head_pred = knn_head_classifier.predict(X_test_head)
knn_limbs_pred = knn_limbs_classifier.predict(X_test_limbs)

knn_shell_accuracy = accuracy_score(y_test_shell, knn_shell_pred)
knn_head_accuracy = accuracy_score(y_test_head, knn_head_pred)
knn_limbs_accuracy = accuracy_score(y_test_limbs, knn_limbs_pred)

print(f"Shell KNN Model Accuracy: {knn_shell_accuracy:.2f}")
print(f"Head KNN Model Accuracy: {knn_head_accuracy:.2f}")
print(f"Limbs KNN Model Accuracy: {knn_limbs_accuracy:.2f}")

print("Shell KNN Classification Report:\n", classification_report(y_test_shell, knn_shell_pred))
print("Head KNN Classification Report:\n", classification_report(y_test_head, knn_head_pred))
print("Limbs KNN Classification Report:\n", classification_report(y_test_limbs, knn_limbs_pred))




Usw sift to contrast

In [ ]:
# Train Decision Tree classifiers for each part
tree_shell_classifier = DecisionTreeClassifier()
tree_head_classifier = DecisionTreeClassifier()
tree_limbs_classifier = DecisionTreeClassifier()

tree_shell_classifier.fit(X_train_shell, y_train_shell)
tree_head_classifier.fit(X_train_head, y_train_head)
tree_limbs_classifier.fit(X_train_limbs, y_train_limbs)

# Evaluate the classifiers
tree_shell_pred = tree_shell_classifier.predict(X_test_shell)
tree_head_pred = tree_head_classifier.predict(X_test_head)
tree_limbs_pred = tree_limbs_classifier.predict(X_test_limbs)

tree_shell_accuracy = accuracy_score(y_test_shell, tree_shell_pred)
tree_head_accuracy = accuracy_score(y_test_head, tree_head_pred)
tree_limbs_accuracy = accuracy_score(y_test_limbs, tree_limbs_pred)

print(f"Shell Decision Tree Model Accuracy: {tree_shell_accuracy:.2f}")
print(f"Head Decision Tree Model Accuracy: {tree_head_accuracy:.2f}")
print(f"Limbs Decision Tree Model Accuracy: {tree_limbs_accuracy:.2f}")

print("Shell Decision Tree Classification Report:\n", classification_report(y_test_shell, tree_shell_pred))
print("Head Decision Tree Classification Report:\n", classification_report(y_test_head, tree_head_pred))
print("Limbs Decision Tree Classification Report:\n", classification_report(y_test_limbs, tree_limbs_pred))

In [ ]:
# Extract SIFT features for shell, head, and limbs
sift = cv2.SIFT_create()

sift_features_dir = os.path.join(data_dir, 'features_sift')
shell_sift_features_dir = os.path.join(sift_features_dir, 'shell')
head_sift_features_dir = os.path.join(sift_features_dir, 'head')
limbs_sift_features_dir = os.path.join(sift_features_dir, 'limbs')
os.makedirs(shell_sift_features_dir, exist_ok=True)
os.makedirs(head_sift_features_dir, exist_ok=True)
os.makedirs(limbs_sift_features_dir, exist_ok=True)

def extract_sift_features(image_path, feature_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        keypoints, descriptors = sift.detectAndCompute(img, None)
        if descriptors is not None:
            np.save(feature_path, descriptors)
            print(f"Extracted SIFT features for {image_path} with {len(keypoints)} keypoints and descriptor shape {descriptors.shape}")
        else:
            print(f"No SIFT descriptors found for {image_path}")
    else:
        print(f"Failed to load image for SIFT extraction: {image_path}")

# Extract SIFT features for shell, head, and limbs
for img_file in os.listdir(shell_output_dir):
    shell_path = os.path.join(shell_output_dir, img_file)
    feature_path = os.path.join(shell_sift_features_dir, f'shell_{img_file}.npy')
    extract_sift_features(shell_path, feature_path)

for img_file in os.listdir(head_output_dir):
    head_path = os.path.join(head_output_dir, img_file)
    feature_path = os.path.join(head_sift_features_dir, f'head_{img_file}.npy')
    extract_sift_features(head_path, feature_path)

for img_file in os.listdir(limbs_output_dir):
    limbs_path = os.path.join(limbs_output_dir, img_file)
    feature_path = os.path.join(limbs_sift_features_dir, f'limbs_{img_file}.npy')
    extract_sift_features(limbs_path, feature_path)

print("SIFT feature extraction completed for all parts.")


In [ ]:
# Combine SIFT features and labels for dataset creation
shell_sift_features = [np.mean(np.load(os.path.join(shell_sift_features_dir, f)), axis=0) for f in os.listdir(shell_sift_features_dir)]
head_sift_features = [np.mean(np.load(os.path.join(head_sift_features_dir, f)), axis=0) for f in os.listdir(head_sift_features_dir)]
limbs_sift_features = [np.mean(np.load(os.path.join(limbs_sift_features_dir, f)), axis=0) for f in os.listdir(limbs_sift_features_dir)]

# Create labels for each part ('S' for shell, 'H' for head, 'L' for limbs)
shell_sift_labels = ['Shell'] * len(shell_sift_features)
head_sift_labels = ['Head'] * len(head_sift_features)
limbs_sift_labels = ['Limbs'] * len(limbs_sift_features)

# Combine features and labels
sift_features = np.vstack((shell_sift_features, head_sift_features, limbs_sift_features))
sift_labels = np.array(shell_sift_labels + head_sift_labels + limbs_sift_labels)

# Split the dataset into training and testing sets
X_train_sift, X_test_sift, y_train_sift, y_test_sift = train_test_split(sift_features, sift_labels, test_size=0.2, random_state=23)

print(f"SIFT Training set size: {len(X_train_sift)}, Testing set size: {len(X_test_sift)}")

In [ ]:
# Train SVM classifier with SIFT features
svm_classifier_sift = SVC(kernel='linear', random_state=23)
svm_classifier_sift.fit(X_train_sift, y_train_sift)

# Make predictions
y_pred_sift = svm_classifier_sift.predict(X_test_sift)

# Evaluate the model
accuracy_sift = accuracy_score(y_test_sift, y_pred_sift)
print(f"SIFT Model accuracy: {accuracy_sift:.2f}")
print("Classification Report for SIFT SVM:\n", classification_report(y_test_sift, y_pred_sift))

In [ ]:
# Train SVM classifier with SIFT features
svm_classifier_sift = SVC(kernel='linear', random_state=23)
svm_classifier_sift.fit(X_train_sift, y_train_sift)

# Make predictions
y_pred_sift = svm_classifier_sift.predict(X_test_sift)

# Evaluate the model
accuracy_sift = accuracy_score(y_test_sift, y_pred_sift)
print(f"SIFT Model accuracy: {accuracy_sift:.2f}")
print("Classification Report for SIFT SVM:\n", classification_report(y_test_sift, y_pred_sift))

In [ ]:
# Train KNN classifier with SIFT features
knn_classifier_sift = KNeighborsClassifier(n_neighbors=3)
knn_classifier_sift.fit(X_train_sift, y_train_sift)

# Make predictions
y_pred_knn_sift = knn_classifier_sift.predict(X_test_sift)

# Evaluate the model
accuracy_knn_sift = accuracy_score(y_test_sift, y_pred_knn_sift)
print(f"KNN Model accuracy with SIFT features: {accuracy_knn_sift:.2f}")
print("Classification Report for KNN with SIFT features:\n", classification_report(y_test_sift, y_pred_knn_sift))


In [ ]:
# Train Bayesian (Naive Bayes) classifier with SIFT features
nb_classifier_sift = GaussianNB()
nb_classifier_sift.fit(X_train_sift, y_train_sift)

# Make predictions
y_pred_nb_sift = nb_classifier_sift.predict(X_test_sift)

# Evaluate the model
accuracy_nb_sift = accuracy_score(y_test_sift, y_pred_nb_sift)
print(f"Naive Bayes Model accuracy with SIFT features: {accuracy_nb_sift:.2f}")
print("Classification Report for Naive Bayes with SIFT features:\n", classification_report(y_test_sift, y_pred_nb_sift))

In [ ]:
# Train Decision Tree classifier with SIFT features
dt_classifier_sift = DecisionTreeClassifier(random_state=23)
dt_classifier_sift.fit(X_train_sift, y_train_sift)

# Make predictions
y_pred_dt_sift = dt_classifier_sift.predict(X_test_sift)

# Evaluate the model
accuracy_dt_sift = accuracy_score(y_test_sift, y_pred_dt_sift)
print(f"Decision Tree Model accuracy with SIFT features: {accuracy_dt_sift:.2f}")
print("Classification Report for Decision Tree with SIFT features:\n", classification_report(y_test_sift, y_pred_dt_sift))

In [ ]:
# Ranking the accuracies for all classifiers
accuracies = {
    'HOG SVM': accuracy,
    'HOG Soft Margin SVM': accuracy_soft_margin,
    'HOG KNN': accuracy_knn,
    'HOG Decision Tree': accuracy_dt,
    'HOG Naive Bayes': accuracy_nb,
    'SIFT SVM': accuracy_sift,
    'SIFT KNN': accuracy_knn_sift,
    'SIFT Naive Bayes': accuracy_nb_sift,
    'SIFT Decision Tree': accuracy_dt_sift
}

sorted_accuracies = sorted(accuracies.items(), key=lambda x: x[1], reverse=True)

print("\nRanking of Classifier Accuracies:")
for rank, (classifier, acc) in enumerate(sorted_accuracies, 1):
    print(f"{rank}. {classifier}: {acc:.2f}")


In [ ]:
import joblib
# Save the trained SVM model
joblib.dump(svm_classifier, 'hog_svm_classifier.pkl')
print("SVM model saved as 'hog_svm_classifier.pkl'")

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import shutil
import random

data_dir = 'preprocessed'
subsets = ['train', 'valid', 'test']

lower_red1, upper_red1 = np.array([0, 50, 50]), np.array([10, 255, 255])
lower_red2, upper_red2 = np.array([170, 50, 50]), np.array([180, 255, 255])
lower_blue, upper_blue = np.array([100, 50, 50]), np.array([130, 255, 255])
lower_green, upper_green = np.array([40, 50, 50]), np.array([80, 255, 255])

existence_threshold = 0

# Create directories for extracted parts
shell_output_dir = os.path.join(data_dir, 'extracted', 'shell')
head_output_dir = os.path.join(data_dir, 'extracted', 'head')
limbs_output_dir = os.path.join(data_dir, 'extracted', 'limbs')
os.makedirs(shell_output_dir, exist_ok=True)
os.makedirs(head_output_dir, exist_ok=True)
os.makedirs(limbs_output_dir, exist_ok=True)

# Create directory for new dataset with valid images and 20% of invalid images
new_dataset_dir = os.path.join(data_dir, 'new_dataset')
os.makedirs(new_dataset_dir, exist_ok=True)

valid_images = []
invalid_images = []

total_valid_count = 0
total_invalid_count = 0
for subset in subsets:
    mask_dir = os.path.join(data_dir, subset, 'masks')
    image_dir = os.path.join(data_dir, subset, 'images')
    mask_files = sorted(os.listdir(mask_dir))
    image_files = sorted(os.listdir(image_dir))

    # Check if both directories have the same number of files

    if len(mask_files) != len(image_files):
        print(f"Warning: Number of masks and images do not match in {subset} subset.")
        continue

    valid_count = 0
    invalid_count = 0

    for mask_file, image_file in zip(mask_files, image_files):
        mask_path = os.path.join(mask_dir, mask_file)
        image_path = os.path.join(image_dir, image_file)
        
        mask = cv2.imread(mask_path)
        image = cv2.imread(image_path)
        
        if mask is None or image is None:
            print(f"Error reading {mask_file} or {image_file}, skipping.")
            continue

        mask_hsv = cv2.cvtColor(mask, cv2.COLOR_BGR2HSV)
        
        red_mask1 = cv2.inRange(mask_hsv, lower_red1, upper_red1)
        red_mask2 = cv2.inRange(mask_hsv, lower_red2, upper_red2)
        red_mask = cv2.bitwise_or(red_mask1, red_mask2)  
        
        blue_mask = cv2.inRange(mask_hsv, lower_blue, upper_blue)
        green_mask = cv2.inRange(mask_hsv, lower_green, upper_green)
        
        red_count = cv2.countNonZero(red_mask)
        blue_count = cv2.countNonZero(blue_mask)
        green_count = cv2.countNonZero(green_mask)
        
        if red_count > existence_threshold and blue_count > existence_threshold and green_count > existence_threshold:
            valid_images.append(image_path)
            valid_count += 1
        else:
            invalid_images.append(image_path)
            invalid_count += 1

    total_valid_count += valid_count
    total_invalid_count += invalid_count

    print(f"Subset: {subset}, Valid Images: {valid_count}, Invalid Images: {invalid_count}")

# Calculate the total number of images in extracted parts directories
shell_files = len(os.listdir(shell_output_dir))
head_files = len(os.listdir(head_output_dir))
limbs_files = len(os.listdir(limbs_output_dir))

total_extracted_files = shell_files + head_files + limbs_files
print(f"Total Images in Extracted Directories: {total_extracted_files}")
print(f"Valid Images: {total_valid_count}, Invalid Images: {total_invalid_count}")

# Copy valid images to new dataset directory
for image_path in valid_images:
    shutil.copy(image_path, new_dataset_dir)

# Randomly select 20% of invalid images and copy them to new dataset directory
num_invalid_to_copy = int(0.2 * len(invalid_images))
selected_invalid_images = random.sample(invalid_images, num_invalid_to_copy)
for image_path in selected_invalid_images:
    shutil.copy(image_path, new_dataset_dir)

print(f"New dataset created with {len(valid_images)} valid images and {num_invalid_to_copy} invalid images.")



In [ ]:
for idx, img_file in enumerate(os.listdir(new_dataset_dir)):
    if idx >= 3:
        break
    image_path = os.path.join(new_dataset_dir, img_file)
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error reading image: {img_file}, skipping.")
        continue

    # Extract HOG features for the entire image
    img_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    features, _ = hog(img_gray, orientations=9, pixels_per_cell=(8, 8),
                      cells_per_block=(2, 2), block_norm='L2-Hys', visualize=True)
    features = features.reshape(1, -1)

    # Predict the part of the turtle
    prediction = svm_classifier.predict(features)[0]

    mask = np.zeros_like(image)

    # Apply the prediction to create an annotated mask
    if prediction == 'Shell':
        mask[:, :] = [0, 255, 0]  # Green for shell
    elif prediction == 'Head':
        mask[:, :] = [255, 0, 0]  # Red for head
    elif prediction == 'Limbs':
        mask[:, :] = [0, 0, 255]  # Blue for limbs

    # Combine the original image with the mask
    annotated_image = cv2.addWeighted(image, 0.3, mask, 0.5, 0)

    # Display the original and annotated images side by side
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB))
    plt.title('Annotated Image')
    plt.axis('off')

    plt.show()

print("Annotation of the first three images completed.")

